# Notebook 05 — Evaluate Wild-Type Generated Molecules

Day 5–6 goals:
1. Validity / Uniqueness / Diversity metrics
2. QED / SA Score / Lipinski
3. AutoDock Vina scoring for all valid molecules
4. Generate distributions and preliminary Top-20 list


In [ ]:
import os, sys, json
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)

from src import MolEvaluator, evaluate_batch, VinaDocker, dock_molecules, load_config
from src.utils import sdf_to_smiles_list, filter_top_candidates
import pandas as pd

cfg = load_config(f'{PROJECT_ROOT}/configs/targets.yaml')
samp_cfg = load_config(f'{PROJECT_ROOT}/configs/sampling.yaml')

SDF_FILE = f'{PROJECT_ROOT}/results/generated/1M17_raw.sdf'
VINA_CSV = f'{PROJECT_ROOT}/results/vina_scores/1M17_vina.csv'
METRICS_CSV = f'{PROJECT_ROOT}/results/vina_scores/1M17_metrics.csv'

In [ ]:
# ── Step 1: Drug-likeness metrics ───────────────────────────────────────────
smiles_list = sdf_to_smiles_list(SDF_FILE)
print(f'Total molecules in SDF: {len(smiles_list)}')

metrics_df = evaluate_batch(smiles_list, output_csv=METRICS_CSV)

In [ ]:
# ── Step 2: Vina docking ────────────────────────────────────────────────────
with open(f'{PROJECT_ROOT}/data/pocket_centers.json') as f:
    centers = json.load(f)

docker = VinaDocker(
    receptor_pdbqt=f'{PROJECT_ROOT}/data/pdb/1M17_receptor.pdbqt',
    center=tuple(centers['1M17']),
    exhaustiveness=samp_cfg['evaluation']['vina_exhaustiveness'],
)

# This step is slow (3–8h for 1000 molecules); use Colab overnight
vina_df = dock_molecules(SDF_FILE, docker, VINA_CSV)

In [ ]:
# ── Step 3: Merge and filter Top candidates ──────────────────────────────────
from src.utils import merge_vina_and_metrics

combined = merge_vina_and_metrics(VINA_CSV, METRICS_CSV)
print(f'Combined dataset: {len(combined)} molecules')

e = samp_cfg['evaluation']
top_df = filter_top_candidates(
    combined,
    vina_cutoff=e['vina_cutoff'],
    qed_cutoff=e['qed_cutoff'],
    sa_cutoff=e['sa_cutoff'],
    top_k=e['top_k'],
)
top_df.to_csv(f'{PROJECT_ROOT}/results/vina_scores/1M17_top10.csv', index=False)
top_df[['smiles', 'vina_score', 'qed', 'sa_score', 'MW', 'LogP', 'Lipinski']]

In [ ]:
# ── Step 4: Visualize distribution ─────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

vina_valid = combined['vina_score'].dropna()
axes[0].hist(vina_valid, bins=40, color='#4C72B0', alpha=0.8)
axes[0].set_xlabel('Vina Score (kcal/mol)'); axes[0].set_title('Vina Distribution (1M17 WT)')

axes[1].scatter(combined['qed'], combined['sa_score'], alpha=0.3, s=10, c='#4C72B0')
axes[1].set_xlabel('QED'); axes[1].set_ylabel('SA Score'); axes[1].set_title('QED vs SA Score')

axes[2].hist(combined['qed'].dropna(), bins=30, color='#55A868', alpha=0.8)
axes[2].set_xlabel('QED'); axes[2].set_title('QED Distribution')

plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/results/figures/1M17_distributions.png', dpi=150)
plt.show()